In [ ]:
import pandas as pd
import numpy as numpy
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df_dtypes_pcd = pd.read_parquet('../data/processed/credit_dtypes_pcd.parquet')
display(df_dtypes_pcd.head(5))

### Distribution of the target variable

In [ ]:
counts = df_dtypes_pcd["loan_status"].value_counts().sort_index()
percent = df_dtypes_pcd["loan_status"].value_counts(normalize=True).sort_index() * 100

labels = ["0 = paying back", "1 = not paying back"]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(labels, counts.values, color=["green", "red"])

# Anzahl und prozentuale Beschriftung der Balken hinzufügen
ax.bar_label(bars, labels=[f"{c:,} ({p:.1f}%)" for c, p in zip(counts.values, percent.values)],
             padding=3)

ax.set_title("Distribution of Loan Status")
ax.set_xlabel("Loan Status")
ax.set_ylabel("Number of Loans")
ax.set_ylim(0, counts.max() * 1.15)  # sonst überschneidet sich die Beschriftung mit der oberen Kante des Diagramms
plt.show()

print(percent.round(2))

---
### 3. Outliers in the Numerical Features

A boxplot is created for each numerical feature to detect outliers.
Points outside the whiskers are considered outliers.

Unrealistic values need particular attention, e.g. an age above 100 years or an employment length longer than the person's age.

In [ ]:
num_cols = ["person_age", "person_income", "person_emp_length",
            "loan_amnt", "cb_person_cred_hist_length"]

# --- Boxplots ---
fig, axes = plt.subplots(1, len(num_cols), figsize=(18, 5))

for ax, col in zip(axes, num_cols):
    sns.boxplot(y=df_dtypes_pcd[col], ax=ax,
                flierprops={"alpha": 0.2, "markersize": 3})
    ax.set_title(col)
    ax.set_ylabel("")
    if col == "person_income":
        ax.set_yscale("log")          # sonst drückt der 6-Mio.-Wert die Box platt
    else:
        ax.ticklabel_format(style="plain", axis="y")

fig.suptitle("Ausreißer in den numerischen Merkmalen", fontsize=14)
plt.tight_layout()
plt.show()

# --- Ausreißer-Übersicht (IQR-Regel, wie im Boxplot) ---
rows = []
for col in num_cols:
    s = df_dtypes_pcd[col].dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((s < lower) | (s > upper)).sum()
    rows.append({
        "Merkmal": col,
        "Untere Grenze": lower,
        "Obere Grenze": upper,
        "Min": s.min(),
        "Max": s.max(),
        "Anzahl Ausreißer": n_out,
        "Anteil %": round(n_out / len(s) * 100, 2),
    })

outliers = pd.DataFrame(rows).set_index("Merkmal")
display(outliers.round(1))

# Die extremsten Fälle
display(df_dtypes_pcd.nlargest(5, "person_age")[["person_age", "person_income", "person_emp_length"]])
display(df_dtypes_pcd.nlargest(5, "person_emp_length")[["person_age", "person_emp_length"]])

### ToDos

- Feature Engineering
- Clean Mapping
